# Diffusion UAV collection: preflight + run
This notebook checks that the local codebase is internally consistent before running training.

In [1]:
import importlib, sys, traceback, json, types, pathlib
root = pathlib.Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

mods = ['config','world','obs','env','condition','replay_diffusion','critic_nets','diffusion_core','diffusion_nets','runner_diffusion','agent_diffusion']
for m in mods:
    if m in sys.modules:
        del sys.modules[m]


In [2]:
import config, env
print('config file:', config.__file__)
print('env file   :', env.__file__)
print('has cfg.chunk   ->', hasattr(config.CFG, 'chunk'))
print('has cfg.replay  ->', hasattr(config.CFG, 'replay'))
print('has cfg.train   ->', hasattr(config.CFG, 'train'))
print('env has obs_dim ->', hasattr(env.CollectDataEnv, 'obs_dim'))
print('env has act_dim ->', hasattr(env.CollectDataEnv, 'act_dim'))
assert hasattr(config.CFG, 'chunk'), 'Current config.py is missing diffusion fields like cfg.chunk / cfg.replay / cfg.train.'
assert hasattr(env.CollectDataEnv, 'obs_dim'), 'Current env.py is missing CollectDataEnv.obs_dim().'
assert hasattr(env.CollectDataEnv, 'act_dim'), 'Current env.py is missing CollectDataEnv.act_dim().'
print('Preflight check passed.')

config file: D:\VScode\Diffusion_Policy_UAV_Collection_Problem\config.py
env file   : D:\VScode\Diffusion_Policy_UAV_Collection_Problem\env.py
has cfg.chunk   -> True
has cfg.replay  -> True
has cfg.train   -> True
env has obs_dim -> True
env has act_dim -> True
Preflight check passed.


In [3]:
import importlib, agent_diffusion
importlib.reload(agent_diffusion)
from agent_diffusion import AgentDiffusion

cfg = config.CFG
# small smoke-test settings
cfg.env.n_uavs = 2
cfg.obs.mode = 'lv+uav+topk'
cfg.obs.topk_oldest = 1
cfg.chunk.chunk_len = 4
cfg.chunk.exec_len = 2
cfg.replay.actor_chunk_len = 4
cfg.replay.batch_size_actor = 8
cfg.replay.batch_size_critic = 8
cfg.train.warmup_steps = 20
cfg.train.num_episodes = 1
cfg.train.num_timestep = 8
cfg.train.eval_every = 1
cfg.validate()

agent = AgentDiffusion(cfg, device='cpu')
print('Agent created.')
print('obs_dim =', agent.obs_dim, 'act_dim =', agent.act_dim)

Agent created.
obs_dim = 56 act_dim = 4


In [4]:
agent.warmup()
print('Replay size after warmup =', len(agent.replay))
stats = agent.update()
print(stats)

Replay size after warmup = 20
UpdateStats(critic_loss=27618.091796875, actor_loss=7.1002702713012695, bc_loss=1.2502286434173584, q_loss=58.50041961669922)


In [5]:
ret, buf = agent.evaluate(num_timestep=8, deterministic=True, render=False)
print('eval return =', ret)
print('eval avg max buffer =', buf)

eval return = -2218.1418090820316
eval avg max buffer = 21.25


In [6]:
# optional tiny train
agent.train(num_episodes=1, num_timestep=8, eval_every=1, render_eval=False)

[EP 0000] TrainR=-197.29  EvalR=-149.70  TrainBuf=25.50  EvalBuf=24.38  Critic=892.0074  Actor=6.7806  BC=0.6422  Q=61.3835
